# MorphoCoverage: Workflows Example
---

**MorphoCoverage** asks one question of a microscopy dataset: **did I image enough fields?** It
works on feature tables that have already been extracted (one row per cell, nucleus or patch), and
it treats the **field of view (FOV)** as the unit the microscope samples. Objects within one field
are not independent, so counting cells alone can overstate how well a sample was covered.

This notebook runs one dataset through the four modes of MorphoCoverage, each in a few lines:

| Step | Mode | The question |
| --- | --- | --- |
| 3 | **REPORT** | I have finished imaging. What can I trust, and what can't I? |
| 4 | **OPTIMISE / PLAN** | What should I do next time? |
| 5 | **BENCHMARK** | Which stopping rule needs the fewest fields for the same risk of stopping too early? |
| 6 | **ONLINE** | I am imaging now. Should I acquire another field? |

## Prerequisites
- `morphocoverage` installed in your environment (the first cell installs it on Google Colab).
- Two tables:
  - a **feature table**, one row per object, with an FOV identifier and numeric feature columns;
  - a **FOV table**, one row per acquired field, **including fields where nothing was detected**.
- By default the notebook uses the bundled example dataset, idr0028 screen B: nuclei from 12 wells
  on 3 plates, 28 fields each, with YAP/TAZ nuclear-to-cytoplasmic ratios (CC BY 4.0).

## Tips
- **Declare once.** Every scientific choice (readout, features, controls, tolerances) is made in
  a single `Setup` (Step 2). The workflows add no choice of their own.
- **"Not assessed" is an answer.** If a declaration is missing (for example a positive control),
  the report says which questions it could not answer and what they would need.
- **Planning advice rests on held-back fields.** How many fields to image next time comes from what
  random subsets of fields actually preserved, never from the best subset of this dataset.
- Full documentation: `docs/workflows.md` and `docs/questions.md`.

# Step 0: Initialize Environment & Install Dependencies
---

**Run the following cell to initialize the environment.**

* **For Google Colab:** The notebook will automatically detect the environment, clone the
  repository (which holds the example data) and install MorphoCoverage.

* **For Local Jupyter Lab:** The notebook will skip installation steps to avoid messing up your
  local environment. From a clone of the repository, install once with
  `python -m pip install -e . ipywidgets matplotlib`, then open this notebook from the clone.

> **Colab users:** run this cell first; it installs everything, which takes about a minute.
>
> **JupyterLab users:** make sure the environment above is active; the cell only checks it.

**How the settings work.** Settings are chosen with widgets (text boxes, toggles), which work in
both Colab and JupyterLab. Each settings cell displays its widgets; set them, then run the next
cell, which reads the values. Re-run that next cell after changing a setting.

In [ ]:
# @title  Initialize Environment & Install Dependencies

import sys
from pathlib import Path

# Metadata
current_version = "0.1.0"
notebook_name = "MorphoCoverage_Workflows_Example"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    # %pip installs into this kernel's environment; a regular (not editable) install is
    # importable at once, without restarting the runtime.
    if not Path("/content/MorphoCoverage").exists():  # re-running this cell reuses the clone
        !git clone -q --depth 1 https://github.com/CellMigrationLab/MorphoCoverage /content/MorphoCoverage
    %pip install -q /content/MorphoCoverage

    from google.colab import output
    output.enable_custom_widget_manager()

    data_search_start = Path("/content/MorphoCoverage")
    print("✅ Colab setup done (no runtime restart needed)")

else:
    # Fallback for local environments
    print("✅ Local environment detected. Assuming dependencies are already installed.")
    data_search_start = Path.cwd().resolve()


# ---------------------------------------------------------
# 2. Locate the example data (data/crick_idr in the repository)
# ---------------------------------------------------------

def find_repository_root(start: Path) -> Path | None:
    for folder in [start, *start.parents]:
        if (folder / "data" / "crick_idr" / "datasets.py").exists():
            return folder
    return None


repository_root = find_repository_root(data_search_start)
if repository_root is None:
    print("⚠️  WARNING: The example data (data/crick_idr) was not found. "
          "Use your own CSV files in Step 1.")
else:
    sys.path.insert(0, str(repository_root / "data" / "crick_idr"))
    print(f"✅ Example data found in {repository_root / 'data' / 'crick_idr'}")

import os
import tempfile
import threading
import time

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using "
                      "'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("⚠️  WARNING: matplotlib is not installed. The plots will be skipped "
          "(pip install matplotlib).")

# MorphoCoverage imports
from morphocoverage import validate_input_tables
from morphocoverage.workflows import (
    LiveFolder,
    LiveTables,
    PositiveControl,
    Setup,
    benchmark,
    default_families,
    default_novelty,
    document,
    live_session,
    optimise,
    report,
    write_fov,
)

WIDE = {"description_width": "initial"}
LAYOUT = widgets.Layout(width="70%")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

print("✅ Imports successful.")

In [ ]:
# @title Helper utilities (plots)

def show_design_effect(variability, *, title: str):
    # One bar per analysis group: how much larger the variance of the mean is when fields,
    # not random objects, are the sampling unit (1 = fields behave like random objects).
    if plt is None:
        return
    table = variability.groups.dropna(subset=["design_effect"])
    labels = table[list(variability.group_cols)].astype(str).agg(" / ".join, axis=1)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.bar(labels, table["design_effect"], color="#4C72B0")
    ax.axhline(1, color="0.3", lw=1, ls="--", label="fields behave like random objects")
    ax.set_ylabel("design effect")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=90)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def show_reliability(reliability, *, reference: float = 0.9):
    # How often k random fields preserved each declared criterion in held-back fields:
    # one thin line per group, the median across groups in bold.
    if plt is None:
        return
    curve = reliability.curve
    criteria = list(dict.fromkeys(curve["criterion"]))
    fig, axes = plt.subplots(1, len(criteria), figsize=(5 * len(criteria), 3.5), sharey=True)
    for ax, criterion in zip(np.atleast_1d(axes), criteria):
        rows = curve[curve["criterion"] == criterion]
        for _, group in rows.groupby(list(reliability.group_cols)):
            ax.plot(group["fovs"], group["preserved"], color="0.75", lw=0.8)
        median = rows.groupby("fovs")["preserved"].median()
        ax.plot(median.index, median.values, color="#C44E52", lw=2.5, marker="o", label="median")
        ax.axhline(reference, color="0.3", lw=1, ls="--", label=f"{reference:.0%}")
        ax.set_title(criterion, fontsize=9)
        ax.set_xlabel("random fields acquired")
        ax.set_ylim(-0.02, 1.02)
    np.atleast_1d(axes)[0].set_ylabel("share of draws that preserved it")
    np.atleast_1d(axes)[0].legend(frameon=False, loc="lower right")
    plt.tight_layout()
    plt.show()


def show_frontier(frontier, *, alphas=(0.05, 0.10), title: str = ""):
    # Effort against risk of stopping too early, one line per rule family across its settings.
    if plt is None:
        return
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for family, rows in frontier.curve.groupby("family", sort=False):
        rows = rows.sort_values("effort")
        ax.plot(rows["effort"], rows["risk"], marker="o", ms=3, label=family)
    for alpha in alphas:
        ax.axhline(alpha, color="0.5", lw=0.8, ls=":")
    ax.set_xlabel("fields acquired (mean over replays)")
    ax.set_ylabel("risk of stopping too early")
    ax.set_title(title, fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()


def show_live_trace(updates, *, quantity: str, threshold: float, title: str):
    # The evidence each rule saw after every field, and where it first said stop.
    if plt is None:
        return
    rows = [
        {"rule": r.strategy, "fovs": r.n_fovs, "value": r.evidence.value(quantity),
         "stop": r.decision is not None and r.decision.value == "stop"}
        for u in updates for r in u.results
    ]
    trace = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for rule, rows in trace.groupby("rule", sort=False):
        ax.plot(rows["fovs"], rows["value"], marker="o", ms=3, label=rule)
        first = rows[rows["stop"]].head(1)
        ax.scatter(first["fovs"], first["value"], s=90, facecolors="none", edgecolors="k")
    ax.axhline(threshold, color="0.3", lw=1, ls="--", label=f"threshold {threshold}")
    ax.set_xlabel("fields acquired")
    ax.set_ylabel(quantity)
    ax.set_title(title + "  (circle: first stop)", fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()


print("✅ Helpers defined.")

# Step 1: Load your feature tables
---

### **Select the data**

* `Example: idr0028 screen B`: the bundled example (nuclei, 30 features, readout `YAPTAZ_Ratio`).
* `My own CSV files`: enter the paths to your **feature table** and your **FOV table**.

> **Colab users:** upload the files with the folder icon on the left (or mount Google Drive) and
> paste their paths, for example `/content/features.csv`.
>
> **JupyterLab users:** enter a path relative to this notebook or an absolute path.

**⚠️ The FOV table matters.** It lists every field the microscope acquired, including fields where
segmentation found nothing. Without it, empty fields vanish and the effort is under-counted.

**Column names.** MorphoCoverage follows the MorphoBricks names: `condition`, `repeat`, and
`position_id` (or `image_file` when one file is one field) identify a field. Other names, extra
analysis groups (such as `well`) and the observation unit are set in Step 2.

In [ ]:
# @title Choose the data (then run the next cell)

data_source = widgets.ToggleButtons(
    options=["Example: idr0028 screen B", "My own CSV files"], description="Data:", style=WIDE
)
objects_csv = widgets.Text(value="path/to/features.csv", description="Feature table (CSV):",
                           style=WIDE, layout=LAYOUT)
fovs_csv = widgets.Text(value="path/to/fovs.csv", description="FOV table (CSV):",
                        style=WIDE, layout=LAYOUT)
display(widgets.VBox([data_source, objects_csv, fovs_csv]))

In [ ]:
# @title Load the data

output_dir = Path("morphocoverage_output")
output_dir.mkdir(exist_ok=True)

if data_source.value.startswith("Example"):
    from datasets import load

    dataset = load("idr0028_screenB")
    objects, fovs = dataset.tables
    print(f"✅ Loaded {dataset.name}: {dataset.description}")
else:
    objects = pd.read_csv(Path(objects_csv.value), dtype={"repeat": str})
    fovs = pd.read_csv(Path(fovs_csv.value), dtype={"repeat": str})
    dataset = None
    print(f"✅ Loaded {objects_csv.value} and {fovs_csv.value}")

print(f"Objects: {len(objects):,} rows x {objects.shape[1]} columns")
print(f"Fields:  {len(fovs):,} rows x {fovs.shape[1]} columns")
print(f"Output Directory: {output_dir.absolute()}")
display(fovs.head(3))

# Step 2: Declare the dataset once
---

### **The `Setup`**

Everything the four workflows need to know about the dataset is declared here, once.

**1. What you measure**
Leave a control box empty if you have none; the questions it unlocks are then *Not assessed*.

* `Readout`: the scalar you report (one feature column), for example a nuclear/cytoplasmic ratio.
* `Features`: the representation used to find unusual objects and "new" phenotypes. They are
  **always explicit**: numeric metadata (IDs, stage positions, time) never enters phenotype space.
  For the example they are the 30 `Nuclei_*` and `Cytoplasm_*` columns.

**2. Controls (optional, but they unlock questions)**
* `Negative control`: the condition that defines "ordinary" (e.g. `MOCK`). It normalizes the JCB
  rule and is the reference for unusual objects.
* `Positive control`: a condition that should look different (e.g. `YAP` knock-down). Used to check
  that the features can detect what you care about.

**3. What "enough" means**
* `Tolerances`: the readout's mean within these fractions of the full-acquisition value, judged on
  held-back fields. The default (5% and 2.5%) is the JCB paper's.

**4. Acquisition metadata (optional)**
* `Acquisition order` and `stage coordinates`: when recorded, they allow order and spatial analyses.

**Default stopping rules:** the JCB rule (upper 95% bound of the SD, first tolerance), once counting
objects and once counting fields. Pass `rules=(...)` to assess other rules.

In [ ]:
# @title Set the declaration (then run the next cell)

readout_box = widgets.Text(value="YAPTAZ_Ratio", description="Readout column:", style=WIDE,
                           layout=LAYOUT)
prefixes_box = widgets.Text(value="Nuclei_, Cytoplasm_", description="Feature prefixes:",
                            style=WIDE, layout=LAYOUT)
negative_box = widgets.Text(value="MOCK", description="Negative control condition:", style=WIDE)
positive_box = widgets.Text(value="YAP", description="Positive control condition:", style=WIDE)
tolerance_1_box = widgets.FloatText(value=0.05, step=0.005, description="Tolerance 1:", style=WIDE)
tolerance_2_box = widgets.FloatText(value=0.025, step=0.005, description="Tolerance 2:", style=WIDE)
novelty_box = widgets.Checkbox(value=True, description="Assess novelty (default definition)",
                               indent=False)
jobs_box = widgets.IntText(value=-1, description="Workers (-1 = all CPUs):", style=WIDE)
fov_col_box = widgets.Text(value="", placeholder="automatic: position_id, image_file or fov_id",
                           description="FOV column:", style=WIDE, layout=LAYOUT)
groups_box = widgets.Text(
    value=", ".join(dataset.groups) if dataset is not None else "condition, repeat",
    description="Analysis groups (comma-separated):", style=WIDE, layout=LAYOUT,
)
unit_box = widgets.Text(value=dataset.observation_unit if dataset is not None else "",
                        placeholder="e.g. cell, nucleus, patch", description="Observation unit:",
                        style=WIDE)
display(widgets.VBox([readout_box, prefixes_box, negative_box, positive_box, tolerance_1_box,
                      tolerance_2_box, novelty_box, jobs_box, fov_col_box, groups_box, unit_box]))

In [ ]:
# @title Declare the dataset

readout = readout_box.value.strip()
feature_prefixes = prefixes_box.value
negative_control = negative_box.value.strip()
positive_control = positive_box.value.strip()
tolerance_1, tolerance_2 = tolerance_1_box.value, tolerance_2_box.value
use_novelty = novelty_box.value
n_jobs = jobs_box.value
fov_col = fov_col_box.value.strip() or None
group_cols = tuple(c.strip() for c in groups_box.value.split(",") if c.strip())
observation_unit = unit_box.value.strip() or None

# The feature columns are chosen explicitly by prefix: never "all numeric columns".
prefixes = tuple(p.strip() for p in feature_prefixes.split(",") if p.strip())
features = tuple(c for c in objects.columns if c.startswith(prefixes))

order_col = "acquisition_order" if "acquisition_order" in fovs.columns else None
stage = ("stage_x_um", "stage_y_um") if {"stage_x_um", "stage_y_um"} <= set(fovs.columns) else None

novelty = default_novelty(features) if use_novelty and features else None
# A positive control is judged with the definition of new, against the negative control.
use_positive = bool(positive_control and negative_control and novelty is not None)
if positive_control and not use_positive:
    print("⚠️  The positive control needs a negative control and 'Assess novelty': "
          "it is left out, and question 2 will be 'Not assessed'.")

setup = Setup(
    readout=readout,
    features=features,
    control={"condition": negative_control} if negative_control else None,
    positive=(
        PositiveControl({"condition": positive_control}, fovs_seen=10, pair_cols=("repeat",),
                        planted_objects=5)
        if use_positive else None
    ),
    novelty=novelty,
    tolerances=(tolerance_1, tolerance_2),
    acquisition_order_col=order_col,
    stage_cols=stage,
    fov_col=fov_col,
    group_cols=group_cols,
    observation_unit=observation_unit,
    n_jobs=n_jobs,
)

print(f"✅ Readout: {setup.readout}")
print(f"✅ Features: {len(setup.features)} columns, e.g. {list(setup.features[:3])}")
print(f"✅ Controls: negative={setup.control}, "
      f"positive={setup.positive.selector if setup.positive else None}")
print(f"✅ Analysis groups: {setup.group_cols or 'all fields pooled'}; "
      f"observation unit: {setup.observation_unit or 'not declared'}")
print(f"✅ Tolerances: {setup.tolerances}")
print(f"✅ Stopping rules: {[rule.name for rule in setup.stopping_rules()]}")
if setup.novelty is None:
    print("⚠️  No definition of 'new': the novelty questions will be 'Not assessed'.")

In [ ]:
# @title Look at the acquisition, as MorphoCoverage sees it

# The same validation every analysis runs: group columns kept only in the FOV table are
# attached to the objects, and fields without objects stay in the count.
validated = validate_input_tables(
    objects, fovs=fovs, feature_cols=setup.columns, fov_col=setup.fov_col,
    group_cols=list(setup.group_cols) if setup.group_cols is not None else None,
    observation_unit=setup.observation_unit,
)
fov_key = list(validated.fov_key)
per_field = validated.observations.groupby(fov_key).size().rename("objects").reset_index()
acquisition = validated.fovs[fov_key].merge(per_field, on=fov_key, how="left")
acquisition["objects"] = acquisition["objects"].fillna(0)
by = list(validated.group_cols) or (lambda frame: "all fields")
overview = acquisition.groupby(by).agg(
    fields=("objects", "size"),
    empty_fields=("objects", lambda s: int((s == 0).sum())),
    objects=("objects", "sum"),
    objects_per_field=("objects", "mean"),
)
print(f"✅ {len(overview)} analysis groups; a field is identified by {fov_key}")
display(overview.round(1))

# Step 3: REPORT — what can the finished acquisition be trusted for?
---

### **Run the report**

`report(objects, fovs, setup)` opens with the foundation of `docs/questions.md`, then answers its
report questions in order:

- **Foundation: why count fields, and not only objects?** How much do fields differ from one
  another, compared with random objects (the *design effect*)? Would the JCB rule, applied to
  random objects, have been satisfied with whole fields?
1. **Can these features detect what I care about?** Detection and false-alarm rates on the declared
   positive and negative controls.
2. **Did I image enough?** Whether each stopping rule was satisfied, and after how many fields,
   across acquisition orders.
3. **What is limiting my sampling?** The sampling profile: design effect, where the variance of the
   mean sits, clustering of unusual objects.
6. **Which fields should I look at?** Fields to reopen, each with its reason.

**Runtime:** about a minute on the example with 4 workers.

In [ ]:
# @title Run REPORT

print("Running the report...")
started = time.perf_counter()
report_result = report(objects, fovs, setup)
print(f"✅ Report done in {time.perf_counter() - started:.0f} s\n")

print(report_result.summary())

In [ ]:
# @title Plot: do fields behave like random objects?

show_design_effect(report_result.results["variability"],
                   title=f"Design effect of the mean of {setup.readout}, per group")

# Every analysis stays available for further use, as tables:
print("Analyses:", list(report_result.results))
display(report_result.results["variability"].groups[
    [*report_result.results["variability"].group_cols, "fovs", "observations", "icc",
     "design_effect"]
].round(3))

# Step 4: OPTIMISE / PLAN — what should the next experiment do?
---

### **Plan the next acquisition**

`optimise` holds back a share of each group's fields as the reference, then asks how often *k*
random fields from the rest preserved what you declared (the readout's mean within each tolerance).

**Parameters**
* `Field counts`: the numbers of fields to judge.
* `Reservoir fraction`: the share of each group's fields held back. This is a design choice and
  has **no default**: more held-back fields make the reference steadier but leave fewer to draw from.

**What it answers**
4. **How many fields should I image next time?** How often *k* random fields were enough.
5. **What did the extra fields buy?** When rules first and lastingly said stop, and when the held-back
   fields confirmed it was enough.
7. **More fields or more samples?** (needs a declared `hierarchy`, otherwise *Not assessed*.)
8. **Could I have sampled more efficiently?** Recorded, random and dispersed orders, with stage travel.

It also calibrates each stopping rule: how often its stop came too early.

**⚠️ Retrospective is not prospective.** A few *well-chosen* fields may represent this dataset; a new
experiment that samples fields at random needs more. The advice below rests on random draws.

In [ ]:
# @title Set the planning parameters (then run the next cell)

fov_counts_box = widgets.Text(value="2, 4, 8, 12", description="Field counts:", style=WIDE)
reservoir_box = widgets.FloatText(value=0.35, step=0.05, description="Reservoir fraction:",
                                  style=WIDE)
display(widgets.VBox([fov_counts_box, reservoir_box]))

In [ ]:
# @title Run OPTIMISE

fov_counts = fov_counts_box.value
reservoir_fraction = reservoir_box.value

print("Running the planning analyses...")
started = time.perf_counter()
plan = optimise(
    objects,
    fovs,
    setup,
    fov_counts=[int(k) for k in fov_counts.split(",")],
    reservoir_fraction=reservoir_fraction,
)
print(f"✅ Plan done in {time.perf_counter() - started:.0f} s\n")

print(plan.summary())

In [ ]:
# @title Plot: how often do k random fields preserve the mean?

show_reliability(plan.results["reliability"])

### **One report with every question**

`document(report_result, plan)` merges the report and the plan into one document, question by
question, as a microscopist would read it.

In [ ]:
# @title Merge REPORT and PLAN

full_report = document(report_result, plan)
text = full_report.summary()
(output_dir / "report.txt").write_text(text)
print(text)
print(f"\n✅ Saved: {output_dir / 'report.txt'}")

# Step 5: BENCHMARK — which rule needs the fewest fields for the same risk?
---

### **Compare stopping rules at matched risk**

Each rule family is replayed over a grid of thresholds. For every threshold, the benchmark measures
the **effort** (fields acquired before the rule said stop) and the **risk** (how often that stop was
too early, judged on held-back fields). Families are then compared at the same risk, against a
fixed number of fields.

**Default families** (the locked benchmark protocol, version 2): the JCB rule and the JCS descriptor
spread, each counting objects and counting fields, and the FOV bootstrap of the mean.

**⚠️ Development and held-out data.** Choosing thresholds on a dataset and scoring them on the same
dataset is optimistic. Tune on development data and score on held-out data (`docs/benchmarking.md`).
A rule that does no better than a fixed number of fields is a valid result.

**Runtime:** one to two minutes on the example with 4 workers.

In [ ]:
# @title Set the benchmark parameters (then run the next cell)

n_boot_box = widgets.IntText(value=500, description="Bootstrap resamples:", style=WIDE)
display(n_boot_box)

In [ ]:
# @title Run BENCHMARK

n_boot = n_boot_box.value

print("Running the benchmark...")
started = time.perf_counter()
bench = benchmark(objects, fovs, setup, reservoir_fraction=reservoir_fraction, n_boot=n_boot)
print(f"✅ Benchmark done in {time.perf_counter() - started:.0f} s\n")

print(bench.summary())

In [ ]:
# @title Plot: effort against risk, per rule family

for criterion, frontier in bench.frontiers.items():
    show_frontier(frontier, title=criterion)

# Step 6: ONLINE — continue or stop while the microscope acquires
---

### **Two ways to feed a live session**

`live_session(setup)` returns a session that receives **one field at a time** and decides, after
each one, whether to continue. It never sees future fields. The fields can arrive in two ways:

* **Option A — files on disk (`LiveFolder`).** The acquisition side writes one object file per field
  and appends one line per completed field to a manifest (`fovs.csv`). `write_fov` does both, safely:
  the object file is renamed into place in one step, so a half-written field is never read.
* **Option B — tables in memory (`LiveTables`).** When features are computed live and never written
  to disk, pass the growing tables themselves. Each update reads only the new rows.

Both give exactly the decisions of adding each field by hand.

**File format and chunks:** the field is the chunk; one CSV per field reads in milliseconds
(details in `docs/workflows.md`).

### **Option A: Watch a folder while a simulated microscope writes into it**

A background thread plays the microscope: it writes the fields of one well, one every half second.
The watcher polls the folder and stops once every rule says *stop* for that well.

In [ ]:
# @title Set the simulated microscope (then run the next cell)

delay_box = widgets.FloatText(value=0.5, step=0.1, description="Seconds between fields:",
                              style=WIDE)
display(delay_box)

In [ ]:
# @title Option A: Watch a live folder

delay_between_fields_s = delay_box.value

# The fields are named by the validated FOV key (Step 2), and read with the group columns
# attached, so a position name reused in another group is never mixed in.
group_cols = list(validated.group_cols)
metadata_cols = fov_key + ([order_col] if order_col else [])
observations = validated.observations

# One analysis group, in acquisition order.
first_group = validated.fovs[group_cols].drop_duplicates().iloc[0] if group_cols else None
well = (validated.fovs.loc[(validated.fovs[group_cols] == first_group).all(axis=1)]
        if group_cols else validated.fovs)
well = well.sort_values(order_col) if order_col else well
group_label = first_group.to_dict() if group_cols else "all fields"
print(f"Group: {group_label}  ({len(well)} fields)")


def objects_of(field) -> pd.DataFrame:
    mask = np.logical_and.reduce([observations[c].eq(field[c]) for c in fov_key])
    return observations.loc[mask]


live_folder = Path(tempfile.mkdtemp(prefix="morphocoverage_live_"))


def microscope():
    for _, field in well.iterrows():
        write_fov(live_folder, objects_of(field)[setup.columns], field[metadata_cols].to_dict())
        time.sleep(delay_between_fields_s)


writer = threading.Thread(target=microscope, daemon=True)
writer.start()

print(f"Watching {live_folder} ...\n")
watcher = LiveFolder(live_folder, live_session(setup))
seen = watcher.watch(interval=0.1, timeout=60, until_stop=True)
print(f"\n✅ Stopped after {len(seen)} fields "
      f"(the microscope had {len(well)} planned).")

### **Option B: Tables in memory, growing as features are computed**

Here the features never touch the disk. After each field, its objects and its FOV row are appended
to two tables in memory, and `LiveTables.update` passes the new field to the session.

We feed the whole well, without stopping, to see how the evidence of each rule evolves and where it
first said stop.

**Order matters:** a field's objects must be appended **before or with** its FOV row. The FOV row is
the signal that the field is complete; objects that arrive first wait in a buffer.

In [ ]:
# @title Option B: Feed in-memory tables

live_tables = LiveTables(live_session(setup))
growing_objects = observations.iloc[:0]
growing_fovs = validated.fovs.iloc[:0][metadata_cols]
updates = []

for _, field in well.iterrows():
    # ... features of this field are computed here, then appended:
    growing_objects = pd.concat([growing_objects, objects_of(field)])
    growing_fovs = pd.concat([growing_fovs, field[metadata_cols].to_frame().T])
    updates += live_tables.update(growing_objects, growing_fovs)

print(updates[0].summary())
print("...")
print(updates[-1].summary())

show_live_trace(updates, quantity="averaging_cv", threshold=setup.tolerances[0],
                title=f"Live JCB rule, {group_label}")
print(f"✅ {len(updates)} fields added; mean decision time per field: "
      f"{np.mean([r.compute_seconds for u in updates for r in u.results]) * 1000:.1f} ms")

**Reading the plot.** Both lines are the JCB rule on the same field stream; only the unit it
counts differs. Counting **objects**, it treats every nucleus as independent, so its evidence is
available after the first field. Counting **fields**, it needs at least two fields to see
field-to-field variation. When fields differ more than random objects would (a design effect above
1, Step 3, the Foundation), as in every well of the example, the per-object rule is the optimistic one
and stops first. When fields are more alike than random objects (design effect below 1), the order
can reverse, so read the design effect of your own data before trusting either.

### **Option C: Your own acquisition loop**

To connect a real acquisition, keep the session and call it from your own code. Each call adds one
field and returns one result per rule:

```python
session = live_session(setup)

# after the microscope acquired a field and its features were extracted:
results = session.add_fov(field_objects, {"condition": "MOCK", "repeat": "1A", "position_id": "P017"})
if all(r.decision.value == "stop" for r in results):
    print("Enough fields for this sample")
```

# Step 7: Save the results
---

In [ ]:
# @title Save the results as CSV

saved = {
    "fov_variability.csv": report_result.results["variability"].groups,
    "stopping_rules_stops.csv": report_result.results["sufficiency"].stops,
    "sampling_reliability.csv": plan.results["reliability"].curve,
    "stopping_calibration.csv": plan.results["calibration"].table,
}
for criterion, frontier in bench.frontiers.items():
    safe = "".join(ch if ch.isalnum() else "_" for ch in criterion)[:60]
    saved[f"benchmark_{safe}.csv"] = frontier.scores

for name, table in saved.items():
    table.to_csv(output_dir / name, index=False)
    print(f"✅ Saved: {output_dir / name}")